# Amazon Laptop Scraping

In [2]:
# import libraries
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd



In [3]:
%pip install beautifulsoup4 requests pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
# take the url of the page
url = "https://www.amazon.in/s?k=laptops&crid=3R9WE5K6F9ZLA&sprefix=laptops%2Caps%2C522&ref=nb_sb_noss_2"

In [5]:
# create the request header

headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36"
}


In [6]:
# send the request
response = requests.get(url, headers=headers)

# check if the request was successful
response.status_code

200

In [7]:
# show the content of the page
response.content

b'<!doctype html><html lang="en-in" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->\n<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>\n<!-- sp:end-feature:head-start -->\n<!-- sp:feature:csm:head-open-part1 -->\n\n<script type=\'text/javascript\'>var ue_t0=ue_t0||+new Date();</script>\n<!-- sp:end-feature:csm:head-open-part1 -->\n<!-- sp:feature:cs-optimization -->\n<meta http-equiv=\'x-dns-prefetch-control\' content=\'on\'>\n<link rel="preconnect" href="https://images-eu.ssl-images-amazon.com" crossorigin>\n<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>\n<!-- sp:end-feature:cs-optimization -->\n<!-- sp:feature:csm:head-open-part2 -->\n<script type=\'text/javascript\'>\nwindow.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;\nif (window.ue_ihb === 1) {\n\nvar ue_csm = window,\n    ue_hob = +new Date();\n(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()

In [8]:
# beautiful soup object

bs = BeautifulSoup(response.content, "html.parser")
print(bs)

<!DOCTYPE html>
<html class="a-no-js" data-19ax5a9jf="dingo" lang="en-in"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->
<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link crossorigin="" href="https://images-eu.ssl-images-amazon.com" rel="preconnect"/>
<link crossorigin="" href="https://m.media-amazon.com" rel="preconnect"/>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.st

In [9]:
# create the empty list to store the data
Data = []

In [10]:
list_data = []

for page in range(1, 30):
    params = {"k": "laptops", "page": page}
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    bs = BeautifulSoup(response.text, "html.parser")

    product_containers = bs.select('div[data-component-type="s-search-result"]')

    for product in product_containers:

        title_tag = product.select_one("h2 span")
        if not title_tag:
            continue

        #Step 1:Extract the Title
        title = title_tag.get_text(" ", strip=True)

        # Step 2: Extract the Price
        price_tag = product.select_one("span.a-price-whole")

        # Step 3: Extract the Rating
        rating_tag = product.select_one("span.a-size-small.a-color-base")
        match = re.match(r"([A-Za-z]+)", title)


        #Step 5: Extract the Brand Name
        brand = match.group(1) if match else "Unknown"
        
        #Step 6: Extract the RAM
        match = re.search(r"(\d+GB|RAM\s*(\d+GB)?|DDR\d?\s*(\d+GB)?|LPDDR\d?\s*(\d+GB)?)", title, re.IGNORECASE)
        ram = match.group(1) if match else "N/A"

        #Step 7: Extract the Storage 
        match = re.search(r"(\d+)\s*(GB|TB)\s*(?:SSD|Storage|HDD)", title, re.IGNORECASE)
        ssd_storage = f"{match.group(1)}{match.group(2)}" if match else "N/A"

        #Step 8: Extract the Color
        match = re.search(r"\b(Black|White|Silver|Gray|Grey|Red|Blue|Green|Yellow|Pink|Purple|Gold|Bronze|Rose Gold|Indigo|Glacier)\b", title, re.IGNORECASE)
        color = match.group(1) if match else "N/A"
       
        # Step 9: Extract the processor information
        # Step 7: Extract the processor (Intel, AMD, Apple M/A chip, Snapdragon, MediaTek, etc)
        processor = "N/A"
        # Try to match Apple M series (M1, M2, M3, M4, M5, etc.)
        match = re.search(r"Apple\s+M(\d+)", title, re.IGNORECASE)
        if match:
            processor = f"Apple M{match.group(1)}"
        else:
            # Try to match Apple A series (A18, A17, A16, etc.)
            match = re.search(r"Apple\s+A(\d+)", title, re.IGNORECASE)
            if match:
                processor = f"Apple A{match.group(1)}"
            else:
                # Try other processor keywords
                processor_keywords = ["Intel", "AMD", "Snapdragon", "MediaTek", "Celeron"]
                for keyword in processor_keywords:
                    if re.search(rf"\b{keyword}\b", title, re.IGNORECASE):
                        processor = keyword
                        break

        # Append the extracted data to the list
        list_data.append({
            "title": title,
            "price": price_tag.get_text(strip=True) if price_tag else "N/A",
            "rating": rating_tag.get_text(strip=True) if rating_tag else "N/A",
            "brand": brand,
            "ram": ram,
            "storage": ssd_storage,
            "color": color,
            "processor": processor
            })

In [11]:
# display the data item 

for item in list_data:
    print(f"Title: {item['title']}")
    print(f"Price: {item['price']}")
    print(f"Rating: {item['rating']}")
    print(f"Brand: {item['brand']}")
    print(f"RAM: {item['ram']}")
    print(f"Storage: {item['storage']}")
    print(f"Color: {item['color']}")
    print(f"Processor: {item['processor']}")

    print("-" * 50)

Title: HP Omnibook 3, Snapdragon X Processor 45 Tops (16GB LPDDR5x,512GB SSD) 2K WUXGA, 14''/35.6cm, Win 11, M365*Office 24,Silver,1.42kg, hz0026QU/ hz0024QU, Lighter mini Charger, FHD IR Camera, AI Laptop
Price: 69,990
Rating: 3.4
Brand: HP
RAM: 16GB
Storage: 512GB
Color: Silver
Processor: Snapdragon
--------------------------------------------------
Title: Apple 2026 MacBook Neo 13″ Laptop with A18 Pro chip: Built for AI and Apple Intelligence, Liquid Retina Display, 8GB Unified Memory, 256GB SSD Storage, 1080p FaceTime HD Camera; Citrus
Price: 73,990
Rating: 4.8
Brand: Apple
RAM: 8GB
Storage: 256GB
Color: N/A
Processor: N/A
--------------------------------------------------
Title: Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8GB LPDDR5 Ram, 512 GB SSD PCIe, Windows 11 Lifetime Validity,15.6" FHD Screen, AMD Radeon 610M, Silver, 1 Year Brand Warranty
Price: 44,999
Rating: 4.0
Brand: Lenovo
RAM: 8GB
Storage: 512GB
Color: Silver
Processor: AMD
------------------------------------------

In [12]:
# check the length of the data
print(f"Total number of laptops scraped: {len(list_data)}")

Total number of laptops scraped: 530


In [13]:
# display the match
print(brand)

Apple


In [14]:
df = pd.DataFrame(list_data)
df

,title,price,rating,brand,ram,storage,color,processor
0,"HP Omnibook 3, Snapdragon X Processor 45 Tops ...","69,990",3.4,HP,16GB,512GB,Silver,Snapdragon
1,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.8,Apple,8GB,256GB,N/A,N/A
2,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,"44,999",4.0,Lenovo,8GB,512GB,Silver,AMD
3,"ASUS Vivobook 15, Smartchoice,Intel Core i5 13...","65,990",4.1,ASUS,16GB,512GB,Blue,Intel
4,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.7,Apple,8GB,256GB,Silver,N/A
...,...,...,...,...,...,...,...,...
525,"HP Omnibook 5 OLED (Previously Pavilion), Snap...","84,990",4.0,HP,16GB,1TB,Silver,Snapdragon
526,"Lenovo V14 Intel Core i3 13th Gen 14"" FHD (192...","70,990",3.6,Lenovo,16GB,512GB,Grey,Intel
527,Apple 2026 MacBook Pro Laptop with M5 Max chip...,"4,64,490",5.0,Apple,36GB,2TB,Black,N/A
528,Apple 2026 MacBook Air 13″ Laptop with M5 chip...,"1,72,490",4.7,Apple,16GB,1TB,Blue,N/A


In [16]:
import pandas as pd

raw_df = pd.DataFrame(list_data)

raw_df.to_csv(
    r"C:\Users\hp\OneDrive\Desktop\web_scraping_folder\amazon_raw_data.csv",
    index=False
)

In [21]:
df.head()

,title,price,rating,brand,ram,storage,color,processor
0,"HP Omnibook 3, Snapdragon X Processor 45 Tops ...","69,990",3.4,HP,16GB,512GB,Silver,Snapdragon
1,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.8,Apple,8GB,256GB,N/A,N/A
2,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,"44,999",4.0,Lenovo,8GB,512GB,Silver,AMD
3,"ASUS Vivobook 15, Smartchoice,Intel Core i5 13...","65,990",4.1,ASUS,16GB,512GB,Blue,Intel
4,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.7,Apple,8GB,256GB,Silver,N/A


In [22]:
clean_df = df.copy()

In [ ]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 370 entries, 0 to 369
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      370 non-null    object
 1   price      370 non-null    object
 2   rating     370 non-null    object
 3   brand      370 non-null    object
 4   ram        370 non-null    object
 5   storage    370 non-null    object
 6   color      370 non-null    object
 7   processor  370 non-null    object
dtypes: object(8)
memory usage: 23.3+ KB


In [23]:
clean_df.isnull().sum()

title        0
price        0
rating       0
brand        0
ram          0
storage      0
color        0
processor    0
dtype: int64

In [24]:
clean_df.duplicated().sum()

np.int64(249)

In [25]:
# clean price column 
clean_df["price"]=(
    clean_df["price"]
    .astype(str)
    .str.replace(",","",regex=False)
)
#convert price to numeric 
clean_df["price"]=pd.to_numeric(
    clean_df["price"],
    errors="coerce"
)
# fill missing with  the median price 
clean_df["price"]=clean_df["price"].fillna(clean_df["price"].median())
clean_df["price"].head()




0    69990.0
1    73990.0
2    44999.0
3    65990.0
4    73990.0
Name: price, dtype: float64

In [ ]:
clean_df["price"].head()

0     69990.0
1    342490.0
2    172490.0
3    126790.0
4     63990.0
Name: price, dtype: float64

In [26]:
#clean rating 
#convert rating to numeric 
clean_df["rating"]=pd.to_numeric(
    clean_df["rating"],
    errors="coerce"
    )
clean_df["rating"].head()

0    3.4
1    4.8
2    4.0
3    4.1
4    4.7
Name: rating, dtype: float64

In [61]:
# Handle missing/invalid ratings

# Convert rating column to numeric
clean_df["rating"] = pd.to_numeric(
    clean_df["rating"],
    errors="coerce"
)

# Fill missing/invalid ratings with median
clean_df["rating"] = clean_df["rating"].fillna(
    clean_df["rating"].median()
)


In [47]:
clean_df["ram"].value_counts(dropna=False)

# replace N/A values with unknown 
clean_df["ram"]=clean_df["ram"].replace("N/A","Unknown")
clean_df["ram"].value_counts(dropna=False)

# Replace RAM type values with Unknown
clean_df["ram"] = clean_df["ram"].replace(
    ["DDR4", "DDR5", "DDR7", "LPDDR4", "LPDDR5"],
    "Unknown"
)

# Check RAM values
print(clean_df["ram"].value_counts())

ram
16GB       104
8GB         83
Unknown     42
32GB        15
4GB         12
24GB         9
6GB          9
36GB         4
12GB         1
48GB         1
64GB         1
Name: count, dtype: int64


In [46]:
#clean processor 
clean_df["processor"]=clean_df["processor"].replace(
    "N/A","Unknown"
)

#clean space and capitalization 
clean_df["processor"]=(
    clean_df["processor"]
    .astype(str)
    .str.strip()
    .str.title()
)
clean_df["processor"].value_counts()

processor
Intel         143
Amd            80
Unknown        39
Snapdragon     12
Mediatek        7
Name: count, dtype: int64

In [28]:
# clean text columns
text_columns = [
    "title",
    "brand",
    "ram",
    "storage",
    "color",
    "processor",
]


for col in text_columns:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.strip()
    )

# Replace only RAM with Unknown
clean_df["ram"] = clean_df["ram"].replace("RAM", "Unknown")

clean_df["ram"].value_counts()

ram
16GB       188
8GB        159
24GB        75
Unknown     33
32GB        17
4GB         15
DDR5        12
6GB         10
36GB         8
48GB         6
64GB         2
LPDDR4       1
12GB         1
LPDDR5       1
DDR7         1
DDR4         1
Name: count, dtype: int64

In [40]:
# Convert TB values to GB in the same storage column
def convert_storage(value):
    value = str(value).strip().upper()

    if "TB" in value:
        number = float(value.replace("TB", "").strip())
        return f"{int(number * 1024)}GB"

    return value


# Apply conversion to the existing storage column
clean_df["storage"] = clean_df["storage"].apply(convert_storage)

# Check the result
clean_df["storage"].value_counts()

clean_df.drop(columns=["storage_gb"], inplace=True)

In [42]:
# Replace N/A values with Unknown
clean_df = clean_df.replace("N/A", "Unknown")

# Fill actual missing values with Unknown
clean_df = clean_df.fillna("Unknown")

# Check for missing values
print(clean_df.isnull().sum())

title        0
price        0
rating       0
brand        0
ram          0
storage      0
color        0
processor    0
dtype: int64


In [30]:
# remove duplicate rows 
# remove duplicate records 
clean_df= clean_df.drop_duplicates()
clean_df.duplicated().sum()

np.int64(0)

In [31]:
# check the cleaned dataset 
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 281 entries, 0 to 514
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   title       281 non-null    object 
 1   price       281 non-null    float64
 2   rating      236 non-null    float64
 3   brand       281 non-null    object 
 4   ram         281 non-null    object 
 5   storage     281 non-null    object 
 6   color       281 non-null    object 
 7   processor   281 non-null    object 
 8   storage_gb  250 non-null    float64
dtypes: float64(3), object(6)
memory usage: 22.0+ KB


In [53]:
clean_df.head(10)

,title,price,rating,brand,ram,storage,color,processor
0,"HP Omnibook 3, Snapdragon X Processor 45 Tops ...",69990.0,3.4,HP,16GB,512GB,Silver,Snapdragon
1,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,73990.0,4.8,Apple,8GB,256GB,Unknown,Unknown
2,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,44999.0,4.0,Lenovo,8GB,512GB,Silver,Amd
3,"ASUS Vivobook 15, Smartchoice,Intel Core i5 13...",65990.0,4.1,ASUS,16GB,512GB,Blue,Intel
4,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,73990.0,4.7,Apple,8GB,256GB,Silver,Unknown
5,Apple 2026 MacBook Air 13″ Laptop with M5 chip...,138990.0,4.7,Apple,16GB,512GB,Silver,Unknown
6,"ASUS Vivobook 16,Smartchoice, 13th Gen, Intel ...",67990.0,4.0,ASUS,16GB,512GB,Silver,Intel
7,"Dell 15, Intel Core 13th Gen i5-1334U, FHD, 15...",69990.0,3.8,Dell,16GB,512GB,Grey,Intel
8,"HP OmniBook 5 OLED (Previously Pavilion), Snap...",79990.0,4.1,HP,16GB,512GB,Silver,Snapdragon
9,Apple 2026 MacBook Pro Laptop with M5 Max chip...,464490.0,5.0,Apple,36GB,2048GB,Black,Unknown


In [63]:
clean_df.to_csv(
    r"C:\Users\hp\OneDrive\Desktop\web_scraping_folder\amz2_cleaned_data.csv",
    index=False
)

In [64]:
import os

path = r"C:\Users\hp\OneDrive\Desktop\web_scraping_folder\laptop_cleaned_data.csv"

print(os.path.exists(path))

True
